This tutorial shows how to use `sulp` library to implement user level differential privacy (ULDP) with [torch](https://pytorch.org) models. We will use the [QMNIST](https://github.com/facebookresearch/qmnist) dataset as a toy example.

You should be familiar with differential privacy. If not, [https://programming-dp.com/](https://programming-dp.com/) is a good introductory resource. User level privacy extends the privacy protection from one training example to all training examples that were contributed by one person. Without ULDP, even if inferences cannot be made about individual training examples, inferences about people who are represented by many examples may still be possible.

In [ ]:
import os
import torch
import torch.nn.functional as F
import torchvision
from torchvision.transforms import v2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(device)

## Dataset

> **Why QMNIST?** The NIST digit databases that were used to create the MNIST dataset contained pseudo ids of the people who contributed the handwritten samples. There is some [interesting background information](https://proceedings.neurips.cc/paper/2019/file/51c68dc084cb0b8467eafad1330bce66-Paper.pdf) about this. QMNIST recovers these pseudo ids, among some other things. Is it truly necessary to use differential privacy when training with QMNIST? Because it is a public dataset, the answer is obviously "no". However, it is a lightweight, easy to train dataset that is instantly familiar to many, letting us focus on the tutorial part.

Let's begin by downloading the dataset. `qmnist` is a directory where the data is downloaded, you can change it as needed.

In [ ]:
qmnist_train = torchvision.datasets.QMNIST("qmnist", what="train", download=True, compat=False)

In [ ]:
qmnist_train.data.shape

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(6, 1))
for i in range(4):
    axes[i].imshow(qmnist_train.data[i])

In [ ]:
qmnist_train.classes

Nothing out of the ordinary so far. However, the targets part contains some extra information:

In [ ]:
qmnist_train.targets.shape

In [ ]:
qmnist_train.targets[:4]

The first column (index 0) is the label of the digit, like in MNIST. The third column (index 2) is the pseudo-id of the person contributing the digit. The rest of the columns are related to the NIST databases and we will not use them.

First we create some dataloaders to test training without privacy. The torchvision version requires some light preprocessing, for which we create the transforms pipeline. The `compat=True` parameter tells the dataset class to output only the digit label, when used through a dataloader.

In [ ]:
img_transforms = v2.Compose([v2.PILToTensor(), v2.ToDtype(torch.float32, scale=True)])

In [ ]:
qmnist_train_compat = torchvision.datasets.QMNIST("qmnist", what="train", compat=True,
                            transform=img_transforms)

In [ ]:
train_loader = torch.utils.data.DataLoader(qmnist_train_compat,
                                          batch_size=64,
                                          shuffle=True)

In [ ]:
qmnist_test = torchvision.datasets.QMNIST("qmnist", what="test10k", download=True, compat=True,
                            transform=img_transforms)

In [ ]:
test_loader = torch.utils.data.DataLoader(qmnist_test,
                                          batch_size=64,
                                          shuffle=False)

## Test without privacy

Let's make a small convolutional network, good enough for 28x28 digit recognition.

We also define a simple training function. We don't bother with validation during training, because we are not concerned with creating the best possible model here. The goal is to quickly estimate how the model performs without privacy.

In [ ]:
class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = torch.nn.Conv2d(1, 6, kernel_size=(5, 5))
        self.s2 = torch.nn.MaxPool2d(kernel_size=(2, 2), stride=2)
        self.c3 = torch.nn.Conv2d(6, 16, kernel_size=(5, 5))
        self.s4 = torch.nn.MaxPool2d(kernel_size=(2, 2), stride=2)
        self.c5 = torch.nn.Linear(256, 120)
        self.f6 = torch.nn.Linear(120, 84)
        self.output = torch.nn.Linear(84, 10)

    def forward(self, img):
        x = F.relu(self.c1(img))
        x = self.s2(x)
        x = F.relu(self.c3(x))
        x = self.s4(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.c5(x))
        x = F.relu(self.f6(x))
        return self.output(x)

In [ ]:
def train(dataloader, model, loss_fn, optimizer, device, epochs=10):
    num_batches = len(dataloader)
    model.train()
    for i in range(epochs):
        train_loss = 0
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            pred = model(X)
            loss = loss_fn(pred, y)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        epoch_loss = train_loss / num_batches
        print(f"epoch: {i + 1}/{epochs} loss: {epoch_loss:.3f}")

Train the model with the usual NLL loss, because the task is multiclass classification.

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
model = Net().to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
train(train_loader, model, loss_fn, optimizer, device=device)

For testing, create a function that computes all the predictions for a model. We can then plot a confusion matrix for a visual assesment of the model's performance

In [ ]:
def test(dataloader, model, loss_fn, device):
    model.eval()
    all_pred = torch.tensor((), device=device)
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            pred = torch.sigmoid(logits)
            all_pred = torch.cat((all_pred, pred), 0) 
    return all_pred.cpu().numpy()

In [ ]:
yhat = test(test_loader, model, loss_fn, device=device)

In [ ]:
_ = ConfusionMatrixDisplay.from_predictions(qmnist_test.targets[:, 0], yhat.argmax(axis=1), labels=np.arange(10),
                              display_labels=qmnist_test.classes,
                              xticks_rotation="vertical")

## User level privacy

We have a dataset and a model that has reasonably good multiclass accuracy. We will now repeat the training with user level privacy. To do so:

1. We define the privacy budget
2. Create a dataloader that does Poisson sampling from the dataset
3. Use the `sulp` library to manage the gradient during model training


### Privacy accounting

In [ ]:
import sulp
import opacus  # for DP accounting

The ($\varepsilon$, $\delta$) privacy budget can be freely chosen. q is the sampling rate and can also be freely chosen. Generally, smaller sampling rate allows training the model with less noise, but makes the gradient estimate less accurate.

In [ ]:
epsilon = 10.0
delta = 0.0000001
q = 0.02

If you're used to epochs, then number of steps = epochs * number of batches in an epoch. Or, the total number of model updates during training. Here we select the number of steps to match the 10 epochs we used with the nonprivate model.

In [ ]:
steps = 500   # epochs/q

Here we compute the noise ratio with [Opacus](https://github.com/meta-pytorch/opacus). You can also use Google's `dp-accounting` library or any other method of DP accounting, including basic DP theorems, pen and paper. You may also just assign some value like sigma=1 and calculate the spent privacy budget after the training finishes.

In [ ]:
sigma = opacus.accountants.utils.get_noise_multiplier(
                target_epsilon=epsilon,
                target_delta=delta,
                sample_rate=q,
                steps=steps,
                accountant="rdp"
            )

In [ ]:
sigma

### Dataloader with Poisson sampling

First, we add the contributor ids to the dataset by creating a custom dataset class. This way, we can use standard torch dataloader machinery to iterate over the input images with contributor ids attached.

The `sulp.GroupPoissonSampler` requires only the list of contributor ids, _in the same order_ as in the dataset. The sampler will return random indices that correspond to groups of training samples belonging to individual writers.

The end result is a dataloader that iterates over a random sample (a $q$ sized fraction) of the input dataset in chunks of size `batch_size`

In [ ]:
train_X = qmnist_train.data
train_y = qmnist_train.targets[:, 0]
contributor_ids = qmnist_train.targets[:, 2]

In [ ]:
class ULDPDataset(torch.utils.data.Dataset):
    def __init__(self, X, contributor, y, transform=None):
        self.img_labels = y
        self.img = X
        self.contributor = contributor
        self.transform = transform

    def __len__(self):
        return self.img.shape[0]

    def __getitem__(self, idx):
        image = torch.unsqueeze(self.img[idx, :, :], 0)
        label = self.img_labels[idx]
        contr_id = self.contributor[idx]
        if self.transform:
            image = self.transform(image)
        return image, contr_id, label

In [ ]:
qmnist_train_uldp = ULDPDataset(train_X, contributor_ids, train_y, transform=img_transforms)

In [ ]:
sampler = sulp.GroupPoissonSampler(contributor_ids.tolist(), q)

In [ ]:
train_loader_uldp = torch.utils.data.DataLoader(
        qmnist_train_uldp,
        batch_size=64,
        sampler=sampler
)

### Training

Training resembles a standard pytorch loop, with the following changes:
- the dataloader iterates over a random selection of persons, instead of the entire dataset
- instead of predictions and loss, a model call computes the per-example gradient
- this gradient is accumulated by the `sulp.GradAccumulator` class that takes care of limiting the per-contributor sensitivity (maximum gradient norm per one person is at most `C`)
- once the current sample is finished, the accumulated gradient + noise scaled to `z` is added to the model

Once the differentially private gradient is transferred to the model, standard PyTorch optimizers can be used to update the model.

In [ ]:
def train_uldp(dataloader, model, loss_fn, optimizer, z, C, qN, device, steps=10):
    model.train()   # modes for layers like dropout

    # gradient accumulator takes the per-sample gradient,
    # uses it to compute the per-user gradient
    # and clips it if necessary
    # C - max gradient norm
    # qN - expected sample size (sample_rate * dataset size)
    ga = sulp.GradAccumulator(model.named_parameters(), C, qN)

    # copies of learnable tensors without autograd
    params, buffers = sulp.detach_params(model)

    # calculate per-sample gradient
    gf = sulp.make_gradient_func(model, loss_fn)

    for i in range(steps):
        for batch, (X, g_id, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            g_id = g_id.to(device)

            sample_grads = gf(params, buffers, X, y)
            ga.accumulate(sample_grads, g_id)

        ga.apply()   # copies grad to model

        # ga.params - only the learnable (required_grad) parameters
        # z - noise ratio
        sulp.add_noise(ga.params, z, C, qN)

        # Param update
        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 50 == 0:
            print(f"{i + 1} steps done.")

`qN` is literally sampling rate $q$ * dataset size $N$. It is the expectation of the size of the sample (the actual sample size varies)

In [ ]:
len(qmnist_train_uldp) * q

The model, loss function and the optimizer do not require any modification.

In [ ]:
dpmodel = Net().to(device)

In [ ]:
optimizer = torch.optim.Adam(dpmodel.parameters(), lr=0.001)

For parameter $C$, or the maximum gradient norm per one person, we select 10.0. This parameter does not affect the privacy budget, but may have a very strong effect on model convergence. There is no rule of thumb for selecting this parameter, although some (conflicting) recommendations exist in scientific papers.

In [ ]:
train_uldp(train_loader_uldp, dpmodel, loss_fn, optimizer, sigma, 10.0, 1200, device, steps=steps)

Let's see how the DP model does. Noise was added during each model update, so normally we expect that it's not quite as good as the nonprivate model.

In [ ]:
yhat = test(test_loader, dpmodel, loss_fn, device=device)

In [ ]:
_ = ConfusionMatrixDisplay.from_predictions(qmnist_test.targets[:, 0], yhat.argmax(axis=1), labels=np.arange(10),
                              display_labels=qmnist_test.classes,
                              xticks_rotation="vertical")

## But I want metrics printed out/logged while I'm training?

True enough, completely blind training may be counterproductive. There is a simple workaround to estimate the training loss or other performance measures by using a separate dataloader. So that training is not slowed down, we take a subsample of the training set:

In [ ]:
subsample = torch.randperm(len(qmnist_train_uldp))[:1024]

In [ ]:
val_dataloader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset((train_X[subsample].float() / 255.0).unsqueeze(dim=1), train_y[subsample]),
    batch_size=64
)

Training function that records history:

In [ ]:
def train_uldp_history(dataloader, model, loss_fn, optimizer, z, C, qN, device, steps,
        val_dataloader):
    model.train()
    history = {"loss": []}

    ga = sulp.GradAccumulator(model.named_parameters(), C, qN)
    params, buffers = sulp.detach_params(model)
    gf = sulp.make_gradient_func(model, loss_fn)

    for i in range(steps):
        for batch, (X, g_id, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)
            g_id = g_id.to(device)

            sample_grads = gf(params, buffers, X, y)
            ga.accumulate(sample_grads, g_id)

        ga.apply()
        sulp.add_noise(ga.params, z, C, qN)

        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 50 == 0:
            val_size = len(val_dataloader.dataset)
            num_batches = len(val_dataloader)
            val_loss = 0
            model.eval()
            with torch.no_grad():
                for X, y in val_dataloader:
                    X, y = X.to(device), y.to(device)
                    logits = model(X)
                    val_loss += loss_fn(logits, y).item()
            val_loss /= num_batches
            print(f"step: {i + 1}/{steps} loss: {val_loss:.3f}")

            history["loss"].append(val_loss)
            model.train()
    return history

Now we can train and observe the progress. Using separate validation data would be even more informative. Although, do keep in mind that **selecting a model checkpoint based on validation loss invalidates the privacy guarantee**. This does not mean that the $\varepsilon$ will shoot to infinity, it will simply increase, but it is important to be aware of this phenomenon.

In [ ]:
dpmodel = Net().to(device)

In [ ]:
optimizer = torch.optim.Adam(dpmodel.parameters(), lr=0.001)

In [ ]:
hist = train_uldp_history(train_loader_uldp, dpmodel, loss_fn, optimizer, sigma, 10, 1200, device,
                500, val_dataloader)

In [ ]:
y = hist["loss"]
x = np.arange(len(y)) + 1
plt.plot(x, y, label="training loss")
plt.ylabel("loss")
plt.xlabel("epochs")
plt.legend()